# Sampling planners on a finite-horizon LQR problem

How much does sampling cost in control accuracy when the optimal sequence is known? This example compares MPPI, the cross-entropy method (CEM), and predictive sampling on the double integrator from GTSAM’s LQR example. Gaussian elimination and a Riccati recursion independently establish the reference. Each sampling method then receives the same rollout budget and random seeds. [Original LQR example](../../../gtsam/examples/linear/LQRExample.ipynb) · [Sampling planner guide](../../../../gtsam/planning/doc/SamplingPlanner.ipynb)

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All rights reserved.  
Authors: Frank Dellaert and Akash Ratheesh. See LICENSE for details.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam_unstable/examples/planning/SamplingPlannerLQR.ipynb)

In [1]:
import sys
if 'google.colab' in sys.modules:
    %pip install --quiet gtsam-develop plotly

In [2]:
import gtsam
import numpy as np
import plotly.graph_objects as go
from gtsam.symbol_shorthand import U, X
from IPython.display import HTML, display
from plotly.subplots import make_subplots

## The graph and its exact solution

The reference graph has four states $x_0,\ldots,x_3$ and three scalar controls. Its code uses $x_{k+1}=Ax_k+Bu_k$ with $A_{12}=0.1$ and $B=[0.005,\,1.0]^\mathsf{T}$. It attaches one state penalty at each of the four times and a second terminal penalty at $x_3$. Thus $Q_f=2Q$ in the equivalent Riccati recursion. On trajectories that satisfy the fixed initial state and dynamics, the GTSAM graph error is

$$E=\tfrac12\sum_{k=0}^{2}(x_k^\mathsf{T}Qx_k+u_k^\mathsf{T}Ru_k)+\tfrac12x_3^\mathsf{T}Q_fx_3.$$

The two graphs below share the same quadratic costs. The Gaussian graph also carries the tight initial prior and the hard dynamics factors. The sampling planner enforces both by starting every rollout at $x_0$ and simulating the dynamics.

In [3]:
horizon = 3
A = np.array([[1.0, 0.1], [0.0, 1.0]])
B = np.array([[0.005], [1.0]])
x0 = np.array([0.0, 1.0])
Q, R = np.eye(2), np.eye(1)
states = [X(k) for k in range(horizon + 1)]
controls = [U(k) for k in range(horizon)]
state_noise = gtsam.noiseModel.Gaussian.Information(Q)
control_noise = gtsam.noiseModel.Gaussian.Information(R)
hard_dynamics = gtsam.noiseModel.Diagonal.Sigmas(np.zeros(2))
initial_noise = gtsam.noiseModel.Isotropic.Sigma(2, 1e-20)

linear = gtsam.GaussianFactorGraph()
linear.add(gtsam.JacobianFactor(states[0], np.eye(2), x0, initial_noise))
cost = gtsam.NonlinearFactorGraph()
for key in [*states, states[-1]]:
    linear.add(gtsam.JacobianFactor(key, np.eye(2), np.zeros(2), state_noise))
    cost.add(gtsam.PriorFactorVector(key, np.zeros(2), state_noise))
for k, key in enumerate(controls):
    linear.add(gtsam.JacobianFactor(key, np.eye(1), np.zeros(1), control_noise))
    cost.add(gtsam.PriorFactorVector(key, np.zeros(1), control_noise))
    linear.add(gtsam.JacobianFactor(states[k + 1], np.eye(2), states[k], -A,
                                    key, -B, np.zeros(2), hard_dynamics))

Reverse-time elimination leaves the tightly anchored initial state until last. This order is numerically stable here with the original initial-prior standard deviation of $10^{-20}$. The Riccati recursion below checks every optimal state and control, as well as the factor of one-half in the reported graph error.

In [4]:
ordering = gtsam.Ordering()
for k in reversed(range(horizon + 1)):
    ordering.push_back(states[k])
    if k:
        ordering.push_back(controls[k - 1])
solution = linear.eliminateSequential(ordering).optimize()
reference_controls = np.array([solution.at(key)[0] for key in controls])
reference_error = linear.error(solution)


P = 2 * Q
feedback = []
for k in reversed(range(horizon)):
    gain = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
    feedback.insert(0, gain)
    P = Q + A.T @ P @ (A - B @ gain)
x = x0.copy()
for k, gain in enumerate(feedback):
    np.testing.assert_allclose(x, solution.at(states[k]), atol=1e-10)
    u = -gain @ x
    np.testing.assert_allclose(u, solution.at(controls[k]), atol=1e-10)
    x = A @ x + B @ u
np.testing.assert_allclose(x, solution.at(states[-1]), atol=1e-10)
np.testing.assert_allclose(0.5 * x0 @ P @ x0, reference_error, atol=1e-10)
print(f"Optimal controls: {reference_controls}")
print(f"Minimum graph error: {reference_error:.10f}")

Optimal controls: [-0.63946054 -0.236538   -0.0831465 ]
Minimum graph error: 0.8442304564


## Matched sampling budget

GTSAM’s vector planner takes a tangent increment and multiplies it by the integration step. Setting the step to one and supplying $(A-I)x+Bu$ reproduces the reference’s discrete transition exactly.

Each solver starts from zero controls and evaluates six batches of 64 candidates. This gives 384 search rollouts per method and seed, plus a rollout of the returned plan. Following Akash Ratheesh’s tuning on separate seeds, vanilla MPPI uses a fixed control standard deviation of 0.1 and inverse temperature of 433. A narrower proposal allows the fixed-spread search to refine a good sequence. CEM starts from unit variance and refits the best eight with a standard deviation floor of 0.02. Predictive sampling starts from unit variance and keeps its best candidate. All three include the incumbent.

In [5]:
drift = A - np.eye(2)

def dynamics(x, u):
    return drift @ x + B @ u

planner = gtsam.SamplingPlannerVectorVector(states, controls, 1.0)
initial = gtsam.Values()
initial.insert(states[0], x0)
for key in controls:
    initial.insert(key, np.zeros(1))

The first metric is $|u_0-u_0^\star|$, the error in the action an MPC controller would execute now. The second is $E-E^\star$, the extra cost over the entire plan. Each plan is also rescored on the original Gaussian graph to verify both graph representations give the same error.

In [6]:
iterations, batch_size = 6, 64
seeds = range(20)
algorithms = {'MPPI': gtsam.MPPI(433.0), 'CEM': gtsam.CEM(8, 0.02),
              'Predictive sampling': gtsam.PredictiveSampling()}
options = gtsam.GaussianProposalOptions()
options.includeNominal = True
metrics = {}
for name, algorithm in algorithms.items():
    measurements = []
    for seed in seeds:
        sigma = 0.1 if name == "MPPI" else 1.0
        proposal = gtsam.GaussianProposal(sigma**2 * np.eye(1), batch_size, seed, options)
        plan = planner.optimize(cost, initial, dynamics, proposal, algorithm, iterations)
        candidate = gtsam.VectorValues()
        for key in states + controls:
            candidate.insert(key, plan.atVector(key))
        error = cost.error(plan)
        np.testing.assert_allclose(error, linear.error(candidate), atol=1e-12)
        assert error >= reference_error - 1e-12
        measurements.append((abs(plan.atVector(controls[0])[0] - reference_controls[0]),
                             error - reference_error))
    metrics[name] = np.array(measurements)

In [7]:
print(f"{'Method':20s} {'Median first-action error':>27s} {'Median excess graph error':>27s}")
for name, samples in metrics.items():
    action_error, excess_error = np.median(samples, axis=0)
    print(f"{name:20s} {action_error:27.7f} {excess_error:27.7f}")

Method                 Median first-action error   Median excess graph error
MPPI                                   0.0084728                   0.0002448
CEM                                    0.0044571                   0.0000515
Predictive sampling                    0.0969566                   0.0429241


Each dot below represents one seed. The box spans the middle half of the runs and its central line marks the median. The logarithmic axes make small errors visible. On this three-control quadratic problem, CEM concentrates its proposals as it refits the elites, while MPPI and predictive sampling retain their original spread. The interactive figure loads Plotly from its version-pinned CDN and requires an internet connection.

In [8]:
figure = make_subplots(rows=1, cols=2, subplot_titles=("First-action absolute error", "Excess graph error"))
for column in range(2):
    for name, samples in metrics.items():
        figure.add_trace(go.Box(name=name, y=samples[:, column], customdata=list(seeds),
                                boxpoints="all", jitter=0.4, showlegend=False,
                                hovertemplate="seed %{customdata}<br>error %{y:.6g}<extra>%{fullData.name}</extra>"),
                         row=1, col=column + 1)
figure.update_yaxes(type="log", title_text="Absolute control error", row=1, col=1)
figure.update_yaxes(type="log", title_text="Graph error above optimum", row=1, col=2)
figure.update_layout(template="plotly_white", height=370, margin=dict(t=42, b=70))
from html import escape

document = figure.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Sampling planners on a finite-horizon LQR problem" style="width:100%;border:0" '
             f'height="{figure.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead



## Tuning the MPPI variants

The three-control LQR problem lets us score each returned trajectory against a known optimum. We select each variant’s spread and weighting parameter by its median excess graph error on seeds 100–105. The original 20 evaluation seeds (0–19) are used only after selection. Each configuration gets 384 candidate trajectories per seed. The two-stage primitives method divides its six batches equally between primitive search and colored MPPI.

The colored proposal uses a spectral exponent of one. With just three time steps its frequency gains are identical, so the long-horizon point-target example illustrates the effect of coloring. The normal-lognormal proposal uses a log-scale standard deviation of 0.4. The smooth proposal samples changes in control, and the one-segment piecewise proposal linearly interpolates the two endpoints. That restriction can leave approximation error on a three-control horizon. Elite MPPI fits the best eight candidates. Tsallis uses shape two and a cost cutoff in graph-error units in place of the exponential inverse temperature. The “Before” column uses unit control standard deviation and inverse temperature 4, or cutoff 1 for Tsallis. The grid includes that starting configuration for every variant.

In [9]:
def variant_plan(method, sigma, weighting, seed):
    covariance = sigma**2 * np.eye(1)
    args = (covariance, batch_size)
    if method == "Primitives":
        tracking = gtsam.MPPITracking(*args, weighting, seed, options)
        return tracking.primitives(
            planner, cost, initial, initial, dynamics, 1, 1.0, 0.0,
            3, 3, 0.0, np.zeros((1, horizon))).nominal

    algorithm = gtsam.MPPI(weighting)
    if method == "Elite":
        algorithm = gtsam.EliteMPPI(8, weighting, 0.02)
    elif method == "Tsallis":
        algorithm = gtsam.TsallisMPPI(weighting, 2.0)
    if method == "Colored":
        proposal = gtsam.ColoredNoiseProposal(*args, 1.0, seed, options)
    elif method == "Normal-lognormal":
        proposal = gtsam.NormalLognormalProposal(*args, 0.4, seed, options)
    elif method == "Smooth":
        proposal = gtsam.SmoothControlProposal(*args, seed, options)
    elif method == "Piecewise":
        proposal = gtsam.PiecewiseLinearProposal(*args, 1, seed, options)
    else:
        proposal = gtsam.GaussianProposal(*args, seed, options)
    return planner.optimize(cost, initial, dynamics, proposal, algorithm, iterations)

def excess_errors(method, sigma, weighting, trial_seeds):
    return np.array([cost.error(variant_plan(method, sigma, weighting, seed)) - reference_error
                     for seed in trial_seeds])

In [10]:
from itertools import product

variant_names = ["Elite", "Tsallis", "Colored", "Normal-lognormal", "Smooth", "Piecewise", "Primitives"]
calibration_seeds = range(100, 106)
selected = {}
for name in variant_names:
    weights = (0.002, 0.02, 0.2, 1.0) if name == "Tsallis" else (4.0, 50.0, 433.0, 1000.0)
    grid = product((0.1, 0.25, 0.5, 1.0), weights)
    selected[name] = min(grid, key=lambda pair: np.median(excess_errors(name, *pair, calibration_seeds)))

In [11]:
variant_errors = {}
baseline_errors = {}
print(f"{'Variant':18s} {'Sigma':>7s} {'Beta or cutoff':>16s} {'Before':>12s} {'Tuned':>12s}")
for name, (sigma, weight) in selected.items():
    starting_weight = 1.0 if name == "Tsallis" else 4.0
    baseline_errors[name] = excess_errors(name, 1.0, starting_weight, seeds)
    variant_errors[name] = excess_errors(name, sigma, weight, seeds)
    print(f"{name:18s} {sigma:7.2f} {weight:16g} {np.median(baseline_errors[name]):12.7f} "
          f"{np.median(variant_errors[name]):12.7f}")

Variant              Sigma   Beta or cutoff       Before        Tuned


Elite                 0.50                4    0.0000438    0.0000276


Tsallis               0.10            0.002    0.0399033    0.0002849


Colored               0.10             1000    0.0213396    0.0002673


Normal-lognormal      0.10              433    0.0203238    0.0001937


Smooth                0.10              433    0.0200639    0.0002324


Piecewise             0.10              433    0.0171570    0.0069966


Primitives            0.25               50    0.0156910    0.0028645


The horizontal distance between each pair of markers shows how much calibration changes the median error on the evaluation seeds. All points use the same 384-rollout budget.

In [12]:
figure = go.Figure()
for name in variant_names:
    before = np.median(baseline_errors[name])
    after = np.median(variant_errors[name])
    figure.add_scatter(x=[before, after], y=[name, name], mode="lines",
                       line=dict(color="#aab6be", width=2), showlegend=False, hoverinfo="skip")
for label, measurements, color, symbol in [
    ("Initial settings", baseline_errors, "#536878", "circle-open"),
    ("Tuned on separate seeds", variant_errors, "#04756f", "diamond"),
]:
    figure.add_scatter(x=[np.median(measurements[name]) for name in variant_names],
                       y=variant_names, mode="markers", name=label,
                       marker=dict(size=10, color=color, symbol=symbol),
                       hovertemplate="%{y}<br>Median excess cost %{x:.6g}<extra>%{fullData.name}</extra>")
figure.update_layout(template="plotly_white", height=395, margin=dict(t=35, l=110),
                     legend=dict(orientation="h", y=1.11),
                     xaxis_title="Median excess graph error on evaluation seeds (lower is better)")
figure.update_xaxes(type="log")


document = figure.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Held-out LQR cost before and after tuning" style="width:100%;border:0" '
             f'height="{figure.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead

